# TIP ESG Platform — Formula Engine
**dss+ | Tire Industry Project**

This notebook documents every ESG formula and lets analysts run live calculations.
It is the Python equivalent of the Excel template's formula cells.

In [ ]:
from dataclasses import dataclass, field
from typing import Optional
import pandas as pd, numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
print("Setup complete")

## 1  Emission Factors
These match the Conversion Tables tab in the Excel template exactly.

In [ ]:
EF = {
    "Natural Gas":             56.1,
    "Coal - Sub bituminous":   96.1,  # Excel X41 hardcoded
    "Coal - Brown briquettes": 97.5,  # Excel X42 hardcoded
    "Coal - Other bituminous": 94.6,  # Excel X43 hardcoded
    "Propane":                 63.1,
    "Fuel Oil":                77.4,
    "Diesel":                  74.1,
    "Petrol":                  69.3,
    "Biomass":                  0.0,  # biogenic excluded
    "Waste tires":             47.5,  # per GJ after heat value
    "LPG":                     56.1,  # Excel X51 hardcoded
    "Other":                   71.9,
}
WASTE_TIRE_HV = 36.23   # GJ per metric T
GJ_TO_MWH     = 1/3.6

df_ef = pd.DataFrame([(k,v) for k,v in EF.items()], columns=["Fuel","EF (T.CO2/GJ LHV)"])
df_ef.set_index("Fuel", inplace=True)
display(df_ef)

## 2  Input Data Model

In [ ]:
@dataclass
class Inputs:
    company: str = ""; year: int = 2023
    # ISO 14001
    total_sites: float = 0; iso_sites: float = 0
    # Production
    production: float = 0       # metric T
    # Water
    water: float = 0            # m3
    # Electricity
    renew_elec: float = 0       # GJ
    nonrenew_elec: float = 0    # GJ
    self_gen: float = 0         # GJ
    steam_p: float = 0          # GJ
    sold_elec: float = 0        # GJ
    sold_steam: float = 0       # GJ
    # Fuels (GJ LHV)
    nat_gas: float = 0
    coal_sub: float = 0; coal_brown: float = 0; coal_other: float = 0
    propane: float = 0
    fuel_oil_a: float = 0; fuel_oil_c: float = 0
    diesel: float = 0; petrol: float = 0; biomass: float = 0
    waste_tires_mt: float = 0   # metric T
    lpg: float = 0; other_fuels: float = 0
    # CO2 inputs
    co2_scope2_steam: float = 0
    co2_scope2_elec: float = 0
    # Waste
    waste_total: float = 0; waste_recovery: float = 0

print("Inputs dataclass ready")

## 3  Calculate Function
Mirrors every Excel formula cell.

In [ ]:
def calculate(d: Inputs) -> dict:
    def sdiv(a,b): return a/b if b else 0

    # ISO 14001
    pct_cert = sdiv(d.iso_sites, d.total_sites)

    # Water KPI  (=IFERROR(R13/R11,"-"))
    water_kpi = sdiv(d.water, d.production)

    # Total Electricity  (=SUM(R17:R19))
    total_elec = d.renew_elec + d.nonrenew_elec + d.self_gen

    # Waste tires heat conversion
    wt_gj = d.waste_tires_mt * WASTE_TIRE_HV

    # Total Energy  (=SUM(R16,R20,R23:R35)-SUM(R21:R22))
    coal_gj  = d.coal_sub + d.coal_brown + d.coal_other
    foil_gj  = d.fuel_oil_a + d.fuel_oil_c
    total_e  = (total_elec + d.steam_p + d.nat_gas + coal_gj + d.propane
                + foil_gj + d.diesel + d.petrol + d.biomass
                + wt_gj + d.lpg + d.other_fuels - d.sold_elec - d.sold_steam)
    e_kpi    = sdiv(total_e, d.production)

    # Scope 1 CO2  (=$X40*AL23/1000 pattern)
    s1_ng  = d.nat_gas      * EF["Natural Gas"]/1000
    s1_coal= coal_gj        * EF["Coal - Sub bituminous"]/1000
    s1_prop= d.propane      * EF["Propane"]/1000
    s1_fo  = foil_gj        * EF["Fuel Oil"]/1000
    s1_die = d.diesel       * EF["Diesel"]/1000
    s1_pet = d.petrol       * EF["Petrol"]/1000
    s1_wt  = wt_gj          * EF["Waste tires"]/1000
    s1_lpg = d.lpg          * EF["LPG"]/1000
    s1_oth = d.other_fuels  * EF["Other"]/1000
    scope1 = s1_ng+s1_coal+s1_prop+s1_fo+s1_die+s1_pet+s1_wt+s1_lpg+s1_oth

    # Scope 2 fallback estimate
    if not d.co2_scope2_elec and d.nonrenew_elec:
        d.co2_scope2_elec = (d.nonrenew_elec * GJ_TO_MWH) * 0.45
    scope2   = d.co2_scope2_steam + d.co2_scope2_elec

    # Totals  (=SUM(AL55,AL53)  =IFERROR(AL57/AL11,""))
    total_co2 = scope1 + scope2
    co2_kpi   = sdiv(total_co2, d.production)

    # Waste
    w_elim   = d.waste_total - d.waste_recovery
    w_rec_pct= sdiv(d.waste_recovery, d.waste_total)
    w_check  = abs(d.waste_total - d.waste_recovery - w_elim) < 1

    return dict(pct_certified=pct_cert, water_kpi=water_kpi,
                total_electricity=total_elec, waste_tires_gj=wt_gj,
                total_energy=total_e, energy_kpi=e_kpi,
                scope1=scope1, scope2=scope2, total_co2=total_co2, co2_kpi=co2_kpi,
                waste_elimination=w_elim, waste_recovery_pct=w_rec_pct,
                check_waste=w_check, check_iso=d.iso_sites<=d.total_sites)

print("calculate() ready")

## 4  Live Demo — VerdaTyres Corp 2023

In [ ]:
inp = Inputs(
    company="VerdaTyres Corp", year=2023,
    total_sites=25, iso_sites=25,
    production=1_515_000,
    water=9_180_000,
    renew_elec=1_045_000, nonrenew_elec=3_804_200, self_gen=1_485,
    steam_p=361_000, sold_elec=1_840,
    nat_gas=6_491_000, coal_other=158_600, propane=102_200,
    fuel_oil_a=3_150, fuel_oil_c=206_500,
    diesel=56_300, petrol=4_320, lpg=505_600, other_fuels=440,
    co2_scope2_steam=10_850,
    waste_total=86_250, waste_recovery=79_710,
)

r = calculate(inp)

summary = {
    "Production (metric T)": f"{inp.production:>15,.0f}",
    "Water intensity (m3/T)": f"{r['water_kpi']:>15.2f}",
    "Total Energy (GJ)":     f"{r['total_energy']:>15,.0f}",
    "Energy KPI (GJ/T)":     f"{r['energy_kpi']:>15.2f}",
    "CO2 Scope 1 (T)":       f"{r['scope1']:>15,.0f}",
    "CO2 Scope 2 (T)":       f"{r['scope2']:>15,.0f}",
    "Total CO2 (T)":         f"{r['total_co2']:>15,.0f}",
    "CO2 intensity (T/T)":   f"{r['co2_kpi']:>15.4f}",
    "Waste recovery %":      f"{r['waste_recovery_pct']*100:>14.1f}%",
    "ISO 14001 certified":   f"{r['pct_certified']*100:>14.0f}%",
    "Waste check":           "OK" if r['check_waste'] else "FAIL",
}
df_sum = pd.DataFrame(list(summary.items()), columns=["KPI","Value"]).set_index("KPI")
display(df_sum)

## 5  Trend Visualisation

In [ ]:
YEARS = list(range(2009,2024))
yrs   = [str(y) for y in YEARS]

# VerdaTyres historical (2009-2023)
ENERGY = [10506,12198,12756,12360,12631,12417,12148,12306,12651,13099,12732,10933,12507,13150,13300]
CO2    = [900,1005,1082,1054,1078,1058,1010,1007,1038,1055,1024,823,842,793,780]
WATER  = [9.43,9.99,10.46,10.24,10.04,9.63,9.64,9.87,9.89,9.73,9.71,8.55,9.25,9.18,9.05]
RENEW  = [0,0,0,0,0,3.1,6.3,9.5,12.4,22,32,43.9,57.3,67.6,69.1]
PROD   = [1289,1507,1622,1513,1563,1554,1523,1562,1589,1680,1660,1310,1462,1515,1515]

e_kpi   = [e/p for e,p in zip(ENERGY,PROD)]
co2_kpi = [c/(p/1000) for c,p in zip(CO2,PROD)]

fig = make_subplots(2,2, subplot_titles=[
    "Total Energy (K GJ)", "Total CO2 (K T)", "Intensity KPIs", "Renewable Elec %"])
kw = dict(mode="lines+markers", marker=dict(size=4))
fig.add_trace(go.Scatter(x=yrs, y=ENERGY, name="Energy", line=dict(color="#00916E",width=2), **kw), 1,1)
fig.add_trace(go.Scatter(x=yrs, y=CO2,    name="CO2",    line=dict(color="#DC2626",width=2), **kw), 1,2)
fig.add_trace(go.Scatter(x=yrs, y=e_kpi,  name="Energy KPI", line=dict(color="#7C3AED",width=2), **kw), 2,1)
fig.add_trace(go.Scatter(x=yrs, y=co2_kpi,name="CO2 KPI", line=dict(color="#EA580C",width=2,dash="dot"), **kw), 2,1)
fig.add_trace(go.Bar(x=yrs, y=RENEW,      name="Renew %", marker_color="rgba(0,145,110,0.7)"), 2,2)
fig.update_layout(height=550, title_text="VerdaTyres Corp — ESG Trends 2009-2023",
    showlegend=True, plot_bgcolor="white", paper_bgcolor="white")
for i in range(1,3):
    for j in range(1,3):
        fig.update_xaxes(tickangle=-45, tickfont=dict(size=8), row=i, col=j)
fig.show()

## 6  Validation Engine

In [ ]:
def validate(inp: Inputs, result: dict, prev_result: dict = None, threshold=20.0):
    flags = []
    if not result["check_waste"]:
        flags.append({"sev":"ERROR","msg":"Waste consistency FAIL",
            "detail":f"Total {inp.waste_total:,.0f} not = Recovery + Elimination"})
    if not result["check_iso"]:
        flags.append({"sev":"ERROR","msg":"ISO sites > total sites","detail":""})
    if prev_result:
        checks = [
            ("Production",    inp.production,      None),
            ("Total Energy",  result["total_energy"], prev_result.get("total_energy")),
            ("Total CO2",     result["total_co2"],    prev_result.get("total_co2")),
        ]
        for name, cur, prev in checks:
            if prev and abs((cur-prev)/max(abs(prev),1)*100) > threshold:
                pct = (cur-prev)/abs(prev)*100
                flags.append({"sev":"WARNING","msg":f"{name}: {pct:+.1f}% YoY variation",
                    "detail":f"Current={cur:,.0f}  Previous={prev:,.0f}"})
    return flags

flags = validate(inp, r)
if not flags:
    print("All checks passed — no flags")
else:
    for f in flags:
        icon = "ERROR" if f["sev"]=="ERROR" else "WARNING"
        print(f"[{icon}] {f['msg']}")
        if f["detail"]: print(f"         {f['detail']}")

## 7  Benchmarking

In [ ]:
BANDS = [
    ("CO2 intensity (T/T)",   r["co2_kpi"],    0.55,0.68,0.82,"T/T",   True),
    ("Energy intensity (GJ/T)",r["energy_kpi"],8.0, 9.2,10.5,"GJ/T",   True),
    ("Water intensity (m3/T)", r["water_kpi"], 5.5, 7.0, 9.0,"m3/T",   True),
    ("Renewable elec %",       inp.renew_elec/max(r["total_electricity"],1)*100,
                                               40,  20,  10,  "%",      False),
    ("Waste recovery %",       r["waste_recovery_pct"]*100,
                                               86,  80,  74,  "%",      False),
]
rows = []
for label, val, q25, med, q75, unit, lb in BANDS:
    if lb:
        pos = "Top 25%" if val<=q25 else ("Above avg" if val<=med else ("Average" if val<=q75 else "Below avg"))
    else:
        pos = "Top 25%" if val>=q75 else ("Above avg" if val>=med else ("Average" if val>=q25 else "Below avg"))
    rows.append({"KPI":label,"Value":round(val,3),"Unit":unit,"Q1 (best)":q25,"Median":med,"Q3":q75,"Position":pos})
display(pd.DataFrame(rows).set_index("KPI"))

# Radar
dims   = [b[0] for b in BANDS] + [BANDS[0][0]]
scores = []
for label,val,q25,med,q75,unit,lb in BANDS:
    rng = max(q75-q25,0.001)
    raw = (val-q25)/rng
    scores.append(max(0,min(100,(1-raw)*100 if lb else raw*100)))
scores += [scores[0]]

fig_r = go.Figure([
    go.Scatterpolar(r=scores, theta=dims, fill="toself", name="VerdaTyres 2023",
        line=dict(color="#00916E",width=2), fillcolor="rgba(0,145,110,0.15)"),
    go.Scatterpolar(r=[65]*len(dims), theta=dims, fill="toself", name="TIP Avg",
        line=dict(color="#9CA3AF",width=1.5,dash="dot"), fillcolor="rgba(156,163,175,0.08)")
])
fig_r.update_layout(polar=dict(radialaxis=dict(range=[0,100])),
    title="VerdaTyres vs TIP Industry Average", height=400)
fig_r.show()
